# 1. Configuración del entorno y carga de componentes

## 1.1. Importación de librerías

Importamos las librerías necesarias para el análisis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import scanpy as sc
import joblib

from scipy.sparse import csr_matrix

# Configuramos el estilo de las visualizaciones
sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

## 1.2. Definición de rutas

Definimos las rutas a los datos de entrada y a las carpetas de salida.

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
FIGURES_PATH = '../outputs/figures/'

# Nombres de los ficheros de entrada
BULK_COUNTS_FILENAME = 'TCGA-LUAD_counts_for_deconvolution.parquet'
BULK_CLINICAL_FILENAME = 'TCGA-LUAD_clinical_for_deconvolution.parquet'
REF_SC_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
#MODEL_FILENAME = 'mlp_marker_genes.joblib'
SPECIFIC_SIGNATURE_FILENAME = 'signature_matrix_topN.tsv'
GLOBAL_SIGNATURE_FILENAME = 'signature_matrix_global_RF.tsv'

os.makedirs(os.path.join(FIGURES_PATH, 'deconvolution'), exist_ok=True)

print("Rutas definidas.")

## 1.3. Carga de datos de TCGA

Cargamos los datos de expresión y clínicos de la cohorte de TCGA-LUAD, que fueron procesados en el notebook anterior.

In [ ]:
print("Cargando datos de TCGA (bulk RNA-seq)...")
bulk_counts_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_COUNTS_FILENAME))
bulk_clinical_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_CLINICAL_FILENAME))

print("Datos de TCGA cargados:")
print(f"  - Matriz de conteos: {bulk_counts_df.shape[0]} muestras x {bulk_counts_df.shape[1]} genes")
print(f"  - Datos clínicos: {bulk_clinical_df.shape[0]} muestras x {bulk_clinical_df.shape[1]} variables")


## 1.4. Carga de datos de referencia

Cargamos el objeto AnnData de scRNA-seq que contiene los perfiles de expresión de referencia para cada tipo celular.

In [ ]:
print("\nCargando datos de referencia (scRNA-seq)...")
adata_ref = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, REF_SC_FILENAME))

print("Datos de referencia cargados:")
print(adata_ref)

## 1.5. Carga de las matrices de firmas 

Cargamos nuestra matriz de firmas general y optimizada, que fueron generadas y validadas en el Notebook 2.

In [ ]:
print("\nCargando la matriz de firmas personalizada...")
specific_signature_matrix = pd.read_csv(
    os.path.join(DATA_PROCESSED_PATH, SPECIFIC_SIGNATURE_FILENAME),
    sep='\t',
    index_col='gene'
)

print("\nCargando la matriz de firmas global...")
global_signature_matrix = pd.read_csv(
    os.path.join(DATA_PROCESSED_PATH, GLOBAL_SIGNATURE_FILENAME),
    sep='\t',
    index_col='gene'
)

# Definimos nuestra lista de tipos celulares de referencia a partir de la firma
cell_types = global_signature_matrix.columns.tolist()

print("Matriz de firmas personalizada cargada:")
print(f"  - Dimensiones: {global_signature_matrix.shape[0]} genes x {global_signature_matrix.shape[1]} tipos celulares")
print(f"  - Tipos celulares en la firma: {cell_types}")
display(global_signature_matrix.head())

# 2. Deconvolución comparativa: Firma personalizada vs. firma estándar


En este apartado, realizamos dos análisis de deconvolución paralelos.

1. **Firma personalizada + EPIC:** Aplicamos nuestra matriz de firmas optimizada utilizando el algoritmo EPIC para obtener una estimación detallada de los 10 tipos celulares de nuestro interés, más una fracción 'otherCells'.
2. **Firma personalizada + EPIC:** Aplicamos nuestra matriz de firmas global utilizando el algoritmo EPIC para obtener una estimación detallada de los 10 tipos celulares de nuestro interés, más una fracción 'otherCells'.
3. **Firma estándar (`quanTIseq`):** Aplicamos el método `quanTIseq` con su firma interna y validada como punto de referencia y comparación.

El objetivo es demostrar la utilidad y el comportamiento de ambos enfoques en esta cohorte específica.

# 2.1. Ejecución de la deconvolución en R

A continuación, se debe ejecutar el script `deconvolution.R`. Con ambas firmas, tanto la global como la personalizada usando los archivos generados en el notebook 2 `signature_matrix_global_RF.tsv` y `signature_matrix_specific_markers.tsv` y guardarlos como `deconv_results_epic_global.csv` y `deconv_results_epic_custom.csv, respectivamente.
Posteriormente se debe ejecutar el script `deconvolution_quantiseq.R`. Este script deconvoluciona los mismos datos usando `quanTIseq` con su firma estándar.

Cada script generará un ficheros CSV de resultados:
- `deconv_results_epic_custom.csv`
- `deconv_results_epic_global.csv`
- `deconv_results_quantiseq.csv`

## 2.2. Carga de los Resultados

Cargamos el fichero CSV generado por el script de R, que contiene las proporciones celulares estimadas para cada muestra por el método EPIC.


In [ ]:
# --- Carga de Resultados de EPIC + Firma Personalizada ---
DECONV_EPIC_CUSTOM_FILENAME = 'deconv_results_epic_custom.csv'
epic_custom_path = os.path.join(DATA_PROCESSED_PATH, DECONV_EPIC_CUSTOM_FILENAME)
try:
    epic_custom_results_df = pd.read_csv(epic_custom_path, index_col='sample').T
    epic_custom_results_df.index.name = 'barcode'
    epic_custom_results_df.columns = epic_custom_results_df.columns.str.replace('.', ' ', regex=False)
    print("Resultados de EPIC con firma personalizada cargados.")
except FileNotFoundError:
    print(f"[ERROR] No se encontró el fichero de resultados de EPIC: {epic_custom_path}")
    epic_custom_results_df = None

In [ ]:
# --- Carga de Resultados de EPIC + Firma global ---
DECONV_EPIC_GLOBAL_FILENAME = 'deconv_results_epic_global.csv'
epic_global_path = os.path.join(DATA_PROCESSED_PATH, DECONV_EPIC_GLOBAL_FILENAME)
try:
    epic_global_results_df = pd.read_csv(epic_global_path, index_col='sample').T
    epic_global_results_df.index.name = 'barcode'
    epic_global_results_df.columns = epic_global_results_df.columns.str.replace('.', ' ', regex=False)
    print("Resultados de EPIC con firma global cargados.")
except FileNotFoundError:
    print(f"[ERROR] No se encontró el fichero de resultados de EPIC: {epic_global_path}")
    epic_global_results_df = None

In [ ]:
DECONV_QUANTISEQ_FILENAME = 'deconv_results_quantisec_new.csv'
quantiseq_path = os.path.join(DATA_PROCESSED_PATH, DECONV_QUANTISEQ_FILENAME)
try:
    # La salida de quanTIseq necesita ser transpuesta
    temp_df = pd.read_csv(quantiseq_path, index_col=0)
    quantiseq_results_df = temp_df.T
    quantiseq_results_df.index.name = 'barcode'
    quantiseq_results_df.columns = quantiseq_results_df.columns.str.replace('.', ' ', regex=False)
    print("Resultados de quanTIseq con firma estándar cargados.")
except FileNotFoundError:
    print(f"[ERROR] No se encontró el fichero de resultados de quanTIseq: {quantiseq_path}")
    quantiseq_results_df = None


## 2.2. Comparación de estrategias de firma personalizada

El primer paso es determinar cuál de nuestras dos firmas personalizadas produce los resultados más fiables al ser usada con el algoritmo EPIC.

In [ ]:
if epic_global_results_df is not None and epic_custom_results_df is not None:
    print("\n--- Comparando la composición promedio de ambos métodos ---")
    
    # Calcular medias
    mean_global = epic_global_results_df.mean()
    mean_custom = epic_custom_results_df.mean()
    
    # Crear un DataFrame para la comparación
    epic_comparison_df = pd.DataFrame({
        'EPIC (Firma global)': mean_global,
        'EPIC (Firma personalizada)': mean_custom
    }).fillna(0)

    # Visualizar
    epic_comparison_df.plot(
        kind='bar',
        figsize=(15, 8),
        title='Comparación de Composición Promedio: EPIC (Global) vs. EPIC (Personalizado)'
    )
    plt.ylabel('Proporción Promedio')
    plt.xlabel('Fracción Celular')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

La comparación revela que la firma basada en marcadores específicos resulta en estimaciones biológicamente inverosímiles, con una supresión casi total de las poblaciones de células T y malignas. En contraste, la firma basada en 500 genes de importancia global produce un perfil de composición más equilibrado y plausible. Por lo tanto, se selecciona la Firma Global como la mejor firma personalizada para la siguiente fase de la comparación.

In [ ]:
epic_results_df = epic_global_results_df

## 2.4. Comparación de la firma personalizada óptima vs. el estándar quanTIseq

In [ ]:
if epic_results_df is not None and quantiseq_results_df is not None:
    print("\n--- Comparando la composición promedio de ambos métodos ---")
    
    # Calcular medias
    mean_epic = epic_results_df.mean()
    mean_quantiseq = quantiseq_results_df.mean()
    
    # Crear un DataFrame para la comparación
    comparison_df = pd.DataFrame({
        'EPIC (Firma Personalizada)': mean_epic,
        'quanTIseq (Firma Estándar)': mean_quantiseq
    }).fillna(0)

    # Visualizar
    comparison_df.plot(
        kind='bar',
        figsize=(15, 8),
        title='Comparación de Composición Promedio: EPIC (Personalizado) vs. quanTIseq (Estándar)'
    )
    plt.ylabel('Proporción Promedio')
    plt.xlabel('Fracción Celular')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 2.5. Decisión Metodológica

La comparación visual muestra una diferencia clave entre los dos métodos. `quanTIseq`, con su firma estándar, estima que una gran parte del tejido (más del 60%) corresponde a células no caracterizadas (`uncharacterized cell`). Esto sugiere que su firma, optimizada para el componente inmunitario, no puede explicar la señal dominante de las células malignas en esta cohorte de tumores.

Por otro lado, nuestro enfoque con **EPIC y una firma global** que incluye una categoría para `malignant cell`, aunque también estima una fracción no explicada (`otherCells`), proporciona una deconvolución más granular y completa del TME.

Por lo tanto, para el resto de los análisis clínicos, **procederemos con los resultados obtenidos de EPIC y nuestra firma global**, ya que ofrecen una visión más informativa y específica para nuestro problema de investigación.

In [ ]:
# Seleccionamos el DataFrame final para el análisis
if epic_results_df is not None:
    deconvolution_results_df = epic_results_df
    print("\nSe ha seleccionado el resultado de EPIC (Firma global) para los análisis posteriores.")

# 3. Análisis Exploratorio de los Resultados (EPIC + Firma Personalizada)

Habiendo seleccionado los resultados de la deconvolución con nuestra firma personalizada como los más informativos, procedemos a su análisis detallado.

## 3.1. Fusión de Resultados con Datos Clínicos

In [ ]:
print("--- Fusionando resultados de EPIC con datos clínicos ---")

# Nos aseguramos de que los índices de ambos DataFrames estén alineados
common_samples = bulk_clinical_df.index.intersection(deconvolution_results_df.index)
analysis_df = pd.concat(
    [bulk_clinical_df.loc[common_samples], deconvolution_results_df.loc[common_samples]],
    axis=1
)

print("Fusión completada.")
print("Dimensiones del DataFrame de análisis final:", analysis_df.shape)
display(analysis_df.head())

## 3.2. Comparación Crítica con las Proporciones de Referencia

Comparamos la composición promedio estimada con las proporciones observadas en el dataset de scRNA-seq. El objetivo no es buscar una coincidencia perfecta, sino entender las diferencias sistemáticas entre las dos cohortes y tecnologías, lo cual es clave para la interpretación de los resultados.

In [ ]:
print("\n--- Comparando las proporciones de EPIC con la referencia scRNA-seq ---")

# Calcular las proporciones reales en el dataset de scRNA-seq de referencia
sc_proportions = adata_ref.obs['cell_type'].value_counts(normalize=True)

# Obtener las proporciones promedio de nuestra deconvolución
# (excluyendo 'otherCells' para una comparación más directa de los tipos definidos)
deconv_proportions = deconvolution_results_df.drop(columns=['otherCells'], errors='ignore').mean()

# Crear un DataFrame combinado
comparison_df = pd.DataFrame({
    'Deconvolución (EPIC)': deconv_proportions,
    'Referencia (scRNA-seq)': sc_proportions
}).fillna(0).sort_values(by='Referencia (scRNA-seq)', ascending=False)

# Visualizar la comparación
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Comparación Crítica: Proporciones de Deconvolución (EPIC) vs. Referencia scRNA-seq', fontsize=18)

# Gráfico de barras
comparison_df.plot(kind='bar', ax=axes[0])
axes[0].set_title('Composición Promedio por Método', fontsize=14)
axes[0].set_ylabel('Proporción Promedio')
axes[0].set_xlabel('Tipo Celular')
axes[0].tick_params(axis='x', rotation=45)

# Scatter plot
sns.regplot(data=comparison_df, x='Referencia (scRNA-seq)', y='Deconvolución (EPIC)', ax=axes[1])
max_val = comparison_df.max().max() * 1.1
axes[1].plot([0, max_val], [0, max_val], 'r--', label='Identidad (y=x)')
axes[1].set_title('Correlación entre Proporciones', fontsize=14)
axes[1].set_xlabel('Proporción en scRNA-seq (Observada)')
axes[1].set_ylabel('Proporción en Deconvolución (Estimada)')
axes[1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("\nAnálisis: Se observa la esperada subestimación de Células T, probablemente debida a sesgos de disociación en la referencia de scRNA-seq. Las proporciones de otras poblaciones, como los fagocitos mononucleares, muestran una mejor concordancia.")

## 3.3. Heterogeneidad en la composición celular
#
Para visualizar la variabilidad entre pacientes, utilizamos un heatmap clusterizado. Esto nos permite identificar si existen subgrupos de tumores con perfiles de infiltración celular similares.

In [ ]:
print("\n--- Visualizando la heterogeneidad inter-tumoral ---")

# Seleccionamos las columnas de proporciones para el heatmap
# Incluimos 'otherCells' para tener la imagen completa
proportions_for_heatmap = analysis_df[deconvolution_results_df.columns]

# Creamos el clustermap
# standard_scale=1 (z-score por columna) ayuda a visualizar qué muestras
# tienen una proporción relativamente alta o baja de un tipo celular.
g = sns.clustermap(
    proportions_for_heatmap,
    cmap='viridis',
    standard_scale=1,
    figsize=(12, 18),
    dendrogram_ratio=0.1,
    yticklabels=False
)
g.fig.suptitle('Heatmap Clusterizado de Proporciones Celulares por Muestra (EPIC)', y=1.02)
plt.show()

## 3.4. Composición Celular Promedio de la Cohorte

Para obtener una visión general, calculamos y visualizamos la proporción promedio de cada tipo celular en toda la cohorte de tumores.

In [ ]:
print("\n--- Visualizando la composición celular promedio (incluyendo otherCells) ---")

# Obtenemos los nombres de todas las fracciones celulares
all_cell_fractions = analysis_df.columns[-11:].tolist()

# Calculamos la media de cada columna de tipo celular
mean_proportions = analysis_df[all_cell_fractions].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x=mean_proportions.index, y=mean_proportions.values)
plt.title('Composición Celular Promedio (con otherCells) en TCGA-LUAD', fontsize=16)
plt.ylabel('Proporción Promedio')
plt.xlabel('Tipo Celular')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(mean_proportions.to_frame(name='Proporción Promedio'))
print("\nAnálisis: La fracción 'otherCells' representa, en promedio, "
      f"{mean_proportions.get('otherCells', 0):.2%} de la composición tisular, "
      "indicando una porción significativa de la señal de expresión que no es explicada por nuestra firma.")


In [ ]:
print(mean_proportions.to_frame(name='Proporción Promedio'))

## 3.5. Correlación con el Estadio del Tumor

Investigamos si la proporción de las fracciones celulares (incluyendo 'otherCells')cambia según el estadio patológico del tumor.

In [ ]:
print("\n--- Correlacionando la composición celular con el estadio del tumor ---")

# Seleccionamos fracciones de interés, incluyendo 'otherCells'
fractions_of_interest = ['T cell', 'mononuclear phagocyte', 'malignant cell', 'fibroblast', 'otherCells']

analysis_df['stage_group'] = analysis_df['ajcc_pathologic_stage'].str.extract(r'(Stage [IV]+)')[0]

fig, axes = plt.subplots(1, len(fractions_of_interest), figsize=(25, 6), sharey=True)
fig.suptitle('Proporción de Fracciones Celulares por Estadio del Tumor', fontsize=16)

for i, cell_type in enumerate(fractions_of_interest):
    sns.boxplot(
        data=analysis_df.dropna(subset=['stage_group']),
        x='stage_group',
        y=cell_type,
        ax=axes[i],
        order=['Stage I', 'Stage II', 'Stage III', 'Stage IV']
    )
    axes[i].set_title(f'Fracción de {cell_type}')
    axes[i].set_xlabel('Estadio del Tumor')
    axes[i].set_ylabel('Proporción Estimada')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 4. Análisis de Supervivencia (Kaplan-Meier)


In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

def plot_kaplan_meier(df, cell_type, ax):
    """Función para generar una curva de Kaplan-Meier para un tipo celular."""
    
    # Crear grupos de "Alta" vs. "Baja" infiltración basado en la mediana
    median_val = df[cell_type].median()
    df['group'] = np.where(df[cell_type] >= median_val, 'Alta', 'Baja')
    
    kmf_alta = KaplanMeierFitter()
    kmf_baja = KaplanMeierFitter()
    
    # Ajustar el modelo a cada grupo
    alta_group = df[df['group'] == 'Alta']
    baja_group = df[df['group'] == 'Baja']
    
    kmf_alta.fit(alta_group['survival_time'], alta_group['event_status'], label=f'Alta {cell_type} (n={len(alta_group)})')
    kmf_baja.fit(baja_group['survival_time'], baja_group['event_status'], label=f'Baja {cell_type} (n={len(baja_group)})')
    
    # Graficar
    kmf_alta.plot_survival_function(ax=ax)
    kmf_baja.plot_survival_function(ax=ax)
    
    # Test estadístico (Log-Rank)
    results = logrank_test(
        alta_group['survival_time'], baja_group['survival_time'],
        event_observed_A=alta_group['event_status'], event_observed_B=baja_group['event_status']
    )
    
    ax.set_title(f'Supervivencia según la fracción de {cell_type}\nLog-Rank p-value: {results.p_value:.3f}')
    ax.set_xlabel('Tiempo (días)')
    ax.set_ylabel('Probabilidad de Supervivencia')
    ax.grid(True)

# Creamos una figura para varias curvas de Kaplan-Meier
fig, axes = plt.subplots(4, 3, figsize=(21, 21))
fig.suptitle('Análisis de Supervivencia de Kaplan-Meier', fontsize=16)

rep_cells = all_cell_fractions

for cell, ax in zip(rep_cells, axes.flatten()):
    plot_kaplan_meier(analysis_df, cell, ax)

for i in range(len(rep_cells), len(axes.flatten())):
    axes.flatten()[i].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

El análisis de supervivencia de Kaplan-Meier muestra que no todas las poblaciones celulares estimadas se asociaban con el pronóstico del paciente. La abundancia total de células T o de células malignas no muestran una correlación significativa con la supervivencia. Sin embargo, se identifican dos componentes clave del microambiente tumoral con un valor pronóstico significativo. Una alta infiltración de fibroblastos (p=0.009) y una alta proporción de células dendríticas plasmocitoides (p=0.031) se asocian de forma estadísticamente significativa con una menor supervivencia, apoyando la hipótesis de que un TME con un estroma denso y un componente inmunitario tolerogénico contribuye a la agresividad de la enfermedad. Adicionalmente, se observó una tendencia (p=0.087) hacia una mejor supervivencia en pacientes con una mayor infiltración de linfocitos B, sugiriendo un posible rol protector de la inmunidad humoral en el adenocarcinoma de pulmón.

Hay una explicación más detallada con artículos que corroboran los datos de infiltración de fibroblastos y de pDC en la memoria